# Real NJ COVID cleaning: 2020–2023

Teacher's requested outcome: aggregate records by postal ZIP and `ILLNESS_ONSET_DATE`, then derive onset weekday.

The encrypted original stays outside GitHub. This notebook decrypts it in memory and exports only aggregates. Missing onset dates are counted separately, never replaced with report dates. The default output folder is beside the original file, outside the repository.

The exported count is an onset-record count after exact full-row deduplication. Infection-episode definitions and repeated-person handling still require confirmation. Do not rename this provisional count to a validated `new_cases` target yet.


## 1. Dependencies

Run the next cell once if the encrypted-workbook reader is missing. Restart the kernel if Jupyter requests it. No password is stored in this notebook.


In [1]:
# Uncomment and run only if these packages are not installed.
# %pip install pandas openpyxl msoffcrypto-tool


In [2]:
%pip install msoffcrypto-tool

Note: you may need to restart the kernel to use updated packages.


## 2. Cleaning functions

ZIP columns are selected by their header names, including the changed 2023 layout. Identifiers preserve leading zeros.

In [3]:
"""Aggregate NJ COVID records by postal ZIP and illness onset date.

Requirements: pandas, openpyxl (date decoding only), msoffcrypto-tool.
The encrypted source is decrypted only in memory. No patient records are exported.
"""
from pathlib import Path
from collections import Counter
from datetime import datetime
from getpass import getpass
from xml.etree import ElementTree as ET
import argparse
import csv
import hashlib
import io
import json
import re
import zipfile

import pandas as pd
import msoffcrypto
from openpyxl.utils.datetime import from_excel, WINDOWS_EPOCH, MAC_EPOCH
from openpyxl.styles.numbers import BUILTIN_FORMATS, is_date_format

SOURCE_DEFAULT = Path(r"C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data\NJ_COVID_2020_2023.xlsx")
REGIONS_DEFAULT = Path(r"C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\regional_attributes_outputs\njny_zcta_regional_attributes_2021.csv")
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}


def normalize_zip(value):
    value = str(value or "").strip()
    if re.fullmatch(r"[0-9]{1,5}", value):
        value = value.zfill(5)
    elif re.fullmatch(r"[0-9]{5}-[0-9]{4}", value):
        value = value[:5]
    return value if re.fullmatch(r"[0-9]{5}", value) and value != "00000" else None


def run(source, regional_reference, output, password, start="2020-01-01", end="2023-12-31"):
    source, regional_reference, output = map(Path, (source, regional_reference, output))
    assert source.is_file() and regional_reference.is_file()
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    assert start <= end
    attrs = pd.read_csv(regional_reference, dtype={"zcta": str})
    nj = set(attrs.loc[attrs.state.eq("NJ"), "zcta"])
    ny = set(attrs.loc[attrs.state.eq("NY"), "zcta"])
    assert nj and ny and not nj.intersection(ny)
    output.mkdir(parents=True, exist_ok=True)

    decrypted = io.BytesIO()
    with source.open("rb") as f:
        office = msoffcrypto.OfficeFile(f)
        office.load_key(password=password, verify_password=True)
        office.decrypt(decrypted)
    del office, password
    decrypted.seek(0)
    print("Workbook unlocked in memory. Reading the four annual sheets.", flush=True)
    z = zipfile.ZipFile(decrypted)
    workbook = ET.fromstring(z.read("xl/workbook.xml"))
    props = workbook.find("m:workbookPr", NS)
    epoch = MAC_EPOCH if props is not None and props.get("date1904") == "1" else WINDOWS_EPOCH
    rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
    targets = {r.get("Id"): r.get("Target") for r in rels}
    sheets = []
    for s in workbook.find("m:sheets", NS):
        rid = s.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
        target = targets[rid]
        member = target.lstrip("/") if target.startswith("/") else "xl/" + target
        sheets.append((s.get("name"), member))
    assert {s for s, _ in sheets} == {"2020", "2021", "2022", "2023"}
    strings = []
    with z.open("xl/sharedStrings.xml") as f:
        root = None
        for event, e in ET.iterparse(f, events=["start", "end"]):
            if root is None:
                root = e
            if event == "end" and e.tag.endswith("}si"):
                strings.append("".join(t.text or "" for t in e.iter() if t.tag.endswith("}t")))
                root.clear()
    styles = ET.fromstring(z.read("xl/styles.xml"))
    custom = {int(e.get("numFmtId")): e.get("formatCode") for e in styles.findall("m:numFmts/m:numFmt", NS)}
    style_is_date = [is_date_format(custom.get(int(e.get("numFmtId")), BUILTIN_FORMATS.get(int(e.get("numFmtId")), ""))) for e in styles.findall("m:cellXfs/m:xf", NS)]
    date_cache = {}

    def parse_date(value, typ, style):
        key = (value, typ, style)
        if key in date_cache:
            return date_cache[key]
        answer = None
        if value:
            try:
                if typ in (None, "n") and style_is_date[style]:
                    answer = from_excel(float(value), epoch).date().isoformat()
                else:
                    for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%m/%d/%Y", "%m/%d/%y", "%d%b%Y", "%Y-%m-%dT%H:%M:%S"):
                        try:
                            answer = datetime.strptime(str(value).strip(), fmt).date().isoformat()
                            break
                        except ValueError:
                            pass
            except (ValueError, TypeError, OverflowError, AttributeError):
                pass
        date_cache[key] = answer
        return answer

    seen_rows = set()
    person_day_zip = set()
    totals = Counter()
    annual = []
    daily = Counter()
    unique_daily = Counter()
    monthly = Counter()
    unmatched_zip = Counter()
    exclusion_counts = Counter()
    global_acceptance = Counter()
    raw_onset_missing_by_sheet = Counter()
    required = {"ZIP", "STATE", "ILLNESS_ONSET_DATE", "DATE_FOR_REPORT", "PCR", "person_id_610"}
    for sheet, member in sheets:
        stats = Counter()
        header = None
        with z.open(member) as f:
            for event, e in ET.iterparse(f, events=["start", "end"]):
                if event == "start" and e.tag.endswith("}sheetData"):
                    sheet_root = e
                if event != "end" or not e.tag.endswith("}row"):
                    continue
                row, metadata = {}, {}
                for cell in e:
                    col = re.sub(r"[0-9]", "", cell.get("r"))
                    typ = cell.get("t")
                    v = cell.find("m:v", NS)
                    value = v.text if v is not None else "".join(t.text or "" for t in cell.iter() if t.tag.endswith("}t"))
                    if typ == "s":
                        value = strings[int(value)]
                    row[col] = value or ""
                    metadata[col] = (typ, int(cell.get("s", "0")))
                if header is None:
                    header = row
                    columns = {name: col for col, name in header.items()}
                    assert required.issubset(columns), f"Missing columns in {sheet}"
                    sheet_root.clear()
                    continue
                if not any(row.values()):
                    sheet_root.clear()
                    continue
                stats["raw_records"] += 1
                record = {name: row.get(col, "") for col, name in header.items()}
                if not record["ILLNESS_ONSET_DATE"]:
                    raw_onset_missing_by_sheet[sheet] += 1
                # Full-row equality only. Repeated IDs are not assumed to be duplicate infections.
                digest = hashlib.blake2b(json.dumps(record, sort_keys=True).encode(), digest_size=16).digest()
                if digest in seen_rows:
                    stats["exact_duplicate_records_removed"] += 1
                    sheet_root.clear()
                    continue
                seen_rows.add(digest)
                stats["records_after_exact_dedup"] += 1
                zip_code = normalize_zip(record["ZIP"])
                state = record["STATE"].strip().upper()
                onset = parse_date(record["ILLNESS_ONSET_DATE"], *metadata.get(columns["ILLNESS_ONSET_DATE"], (None, 0)))
                report = parse_date(record["DATE_FOR_REPORT"], *metadata.get(columns["DATE_FOR_REPORT"], (None, 0)))
                issues = []
                if not zip_code:
                    issues.append("missing_or_invalid_zip")
                elif zip_code in ny:
                    issues.append("zip_matches_ny_reference_state_conflict")
                elif zip_code not in nj:
                    issues.append("zip_not_matched_to_nj_reference")
                if state != "NJ":
                    issues.append("state_not_nj")
                if record["PCR"].strip() != "1":
                    issues.append("pcr_not_confirmed_code_1")
                if not onset:
                    issues.append("missing_onset" if not record["ILLNESS_ONSET_DATE"] else "invalid_onset")
                elif not (start.date().isoformat() <= onset <= end.date().isoformat()):
                    issues.append("onset_outside_analysis_period")
                if onset and report and onset > report:
                    issues.append("onset_after_report")
                for issue in issues:
                    stats["flag_" + issue] += 1
                # A disjoint exclusion combination reconciles records without double counting flags.
                if issues:
                    exclusion_counts[(sheet, ";".join(sorted(issues)))] += 1
                    stats["excluded_records"] += 1
                else:
                    stats["included_onset_records"] += 1
                    daily[(zip_code, onset)] += 1
                    global_acceptance[onset[:4]] += 1
                    pid = record["person_id_610"].strip()
                    if pid:
                        key = hashlib.blake2b(json.dumps([pid, zip_code, onset]).encode(), digest_size=16).digest()
                        if key not in person_day_zip:
                            person_day_zip.add(key)
                            unique_daily[(zip_code, onset)] += 1
                        else:
                            stats["repeated_person_zip_onset_records"] += 1
                    else:
                        stats["included_records_missing_person_id"] += 1
                # Missingness by report month provides a separate descriptive coverage diagnostic.
                if zip_code in nj and state == "NJ" and report:
                    monthly[(zip_code, report[:7], "records")] += 1
                    monthly[(zip_code, report[:7], "with_onset" if onset else "without_onset")] += 1
                if zip_code and zip_code not in nj:
                    unmatched_zip[(zip_code, "NY_reference_match" if zip_code in ny else "unmatched_reference")] += 1
                sheet_root.clear()
                if stats["raw_records"] % 250000 == 0:
                    print(f"Processed {sheet}: {stats['raw_records']:,} records", flush=True)
        assert stats["raw_records"] == stats["exact_duplicate_records_removed"] + stats["records_after_exact_dedup"]
        assert stats["records_after_exact_dedup"] == stats["included_onset_records"] + stats["excluded_records"]
        annual.append({"source_sheet": sheet, **dict(stats), "raw_missing_onset": raw_onset_missing_by_sheet[sheet]})
        totals.update(stats)
        print(f"Finished {sheet}: {stats['included_onset_records']:,} included onset records", flush=True)
    z.close()
    decrypted.close()
    del strings, seen_rows, person_day_zip

    observed = pd.DataFrame([(zz, dd, nn, unique_daily[(zz, dd)]) for (zz, dd), nn in sorted(daily.items())],
                            columns=["zip", "date", "onset_record_count", "unique_person_zip_onset_count"])
    observed["date"] = pd.to_datetime(observed.date)
    observed["onset_day_of_week"] = observed.date.dt.dayofweek + 1
    observed["onset_day_name"] = observed.date.dt.day_name()
    observed["state"] = "NJ"
    observed = observed[["state", "zip", "date", "onset_day_of_week", "onset_day_name", "onset_record_count", "unique_person_zip_onset_count"]]
    assert observed.onset_record_count.sum() == totals["included_onset_records"]
    assert not observed.duplicated(["zip", "date"]).any()
    assert (observed.unique_person_zip_onset_count <= observed.onset_record_count).all()
    observed.to_csv(output / "nj_zip_daily_onset_observed.csv", index=False)

    # ZIP identifiers are not relabeled as ZCTAs. A validated crosswalk is a separate step.
    grid = pd.MultiIndex.from_product([sorted(nj), pd.date_range(start, end)], names=["zip", "date"])
    balanced = observed.set_index(["zip", "date"])[["onset_record_count", "unique_person_zip_onset_count"]].reindex(grid)
    balanced["zero_filled_no_observed_onset_record"] = balanced.onset_record_count.isna()
    for column in ["onset_record_count", "unique_person_zip_onset_count"]:
        balanced[column] = balanced[column].fillna(0).astype("int64")
    balanced = balanced.reset_index()
    balanced.insert(0, "state", "NJ")
    balanced["onset_day_of_week"] = balanced.date.dt.dayofweek + 1
    balanced["onset_day_name"] = balanced.date.dt.day_name()
    balanced["surveillance_coverage_confirmed"] = False
    balanced = balanced[["state", "zip", "date", "onset_day_of_week", "onset_day_name", "onset_record_count", "unique_person_zip_onset_count", "zero_filled_no_observed_onset_record", "surveillance_coverage_confirmed"]]
    assert len(balanced) == len(nj) * len(pd.date_range(start, end))
    assert balanced.onset_record_count.sum() == totals["included_onset_records"]
    balanced.to_csv(output / "nj_zip_daily_onset_balanced_provisional.csv", index=False)

    pd.DataFrame(annual).fillna(0).to_csv(output / "cleaning_summary_by_source_sheet.csv", index=False)
    pd.DataFrame([(s, r, n) for (s, r), n in sorted(exclusion_counts.items())], columns=["source_sheet", "exclusion_reasons", "records"]).to_csv(output / "exclusion_combinations.csv", index=False)
    pd.DataFrame([(zz, g, n) for (zz, g), n in sorted(unmatched_zip.items())], columns=["zip", "reference_status", "records_after_exact_dedup"]).to_csv(output / "zip_reference_review.csv", index=False)
    coverage = []
    for zz, month in sorted({(zz, month) for zz, month, _ in monthly}):
        n = monthly[(zz, month, "records")]
        m = monthly[(zz, month, "without_onset")]
        coverage.append((zz, month, n, n-m, m, m/n))
    pd.DataFrame(coverage, columns=["zip", "report_month", "records_after_exact_dedup", "records_with_parsed_onset", "records_without_parsed_onset", "missing_onset_share"]).to_csv(output / "onset_completeness_by_zip_report_month.csv", index=False)
    summary = {
        "source": str(source), "regional_reference": str(regional_reference),
        "analysis_start": str(start.date()), "analysis_end": str(end.date()),
        "raw_missing_onset_records": sum(raw_onset_missing_by_sheet.values()),
        "raw_missing_onset_share": sum(raw_onset_missing_by_sheet.values()) / totals["raw_records"],
        "totals": dict(totals), "included_records_by_onset_year": dict(global_acceptance),
        "reference_nj_identifiers": len(nj), "observed_zip_identifiers": int(observed.zip.nunique()),
        "observed_region_days": len(observed), "balanced_region_days": len(balanced),
        "calendar_days": len(pd.date_range(start, end)),
        "definitions": {
            "onset_record_count": "Number of eligible source rows after exact full-row deduplication; not yet a validated count of infection episodes.",
            "unique_person_zip_onset_count": "Distinct nonmissing person IDs within each ZIP and onset date; diagnostic alternative, not a reinfection rule.",
            "onset_day_of_week": "Monday=1, Tuesday=2, Wednesday=3, Thursday=4, Friday=5, Saturday=6, Sunday=7.",
            "balanced_zero": "No eligible onset record observed at this ZIP/date. Does not establish zero actual cases or complete surveillance.",
            "zip_match": "Exact identifier match to the existing NJ ZCTA reference. This is not a validated postal ZIP-to-ZCTA crosswalk.",
            "numrecs": "Not summed; source meaning has not been confirmed.",
        },
        "pending_decisions": [
            "Confirm patient/episode semantics and reinfection deduplication with provider.",
            "Confirm geographic and date surveillance coverage, including leading and trailing dates.",
            "Assess onset missingness and changes over time before modeling.",
            "Resolve unmatched ZIPs and validate ZIP-to-ZCTA mapping before regional joins.",
            "Confirm report-date semantics before treating retrospective onset data as available on a forecast date.",
        ],
    }
    (output / "cleaning_manifest.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps({k: summary[k] for k in ["raw_missing_onset_share", "observed_zip_identifiers", "observed_region_days", "balanced_region_days", "calendar_days"]}), flush=True)
    print("All reconciliation checks passed. Outputs:", output.resolve(), flush=True)
    return summary




## 3. Configure paths and run

Update `REGIONAL_REFERENCE` if your repository is in another location. Enter the existing workbook password when prompted. Reading all four annual sheets may take several minutes. Exact NJ reference matches form the provisional analysis subset. NY matches, unmatched identifiers and invalid identifiers are summarized separately, not silently reassigned to NJ.


In [4]:
SOURCE = SOURCE_DEFAULT
REGIONAL_REFERENCE = REGIONS_DEFAULT
OUTPUT = SOURCE.parent / "real_nj_cleaning_outputs"

summary = run(
    SOURCE,
    REGIONAL_REFERENCE,
    OUTPUT,
    password=getpass("Workbook password: "),
    start="2020-01-01",
    end="2023-12-31",
)


Workbook password:  ········


Workbook unlocked in memory. Reading the four annual sheets.
Processed 2020: 250,000 records
Processed 2020: 500,000 records
Finished 2020: 265,371 included onset records
Processed 2021: 250,000 records
Processed 2021: 500,000 records
Processed 2021: 750,000 records
Finished 2021: 437,272 included onset records
Processed 2022: 250,000 records
Processed 2022: 500,000 records
Processed 2022: 750,000 records
Finished 2022: 301,336 included onset records
Finished 2023: 36,415 included onset records
{"raw_missing_onset_share": 0.6065362935885894, "observed_zip_identifiers": 596, "observed_region_days": 324251, "balanced_region_days": 873678, "calendar_days": 1461}
All reconciliation checks passed. Outputs: C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data\real_nj_cleaning_outputs


## 4. Inspect the summaries

`onset_day_of_week` uses Monday=1 through Sunday=7. The observed table contains positive-count ZIP/date groups. The balanced table adds every reference ZIP/date combination, including zero observed records. A zero is not proof of complete surveillance or no actual cases. All coverage flags remain false until source coverage is confirmed.

If sharing the notebook in GitHub, clear all outputs first. Keep real-data CSVs outside the repository. ZIP is still a postal identifier; matching its number to a ZCTA reference does not establish a validated ZIP-to-ZCTA mapping.


In [5]:
annual_summary = pd.read_csv(OUTPUT / "cleaning_summary_by_source_sheet.csv")
display(annual_summary)

daily_onset = pd.read_csv(
    OUTPUT / "nj_zip_daily_onset_observed.csv",
    dtype={"zip": "string"},
    parse_dates=["date"],
)
display(daily_onset.head(20))

print("Raw records:", f"{summary['totals']['raw_records']:,}")
print("Included onset records:", f"{summary['totals']['included_onset_records']:,}")
print("Missing onset share:", f"{summary['raw_missing_onset_share']:.2%}")
print("Balanced rows:", f"{summary['balanced_region_days']:,}")
print("Saved to:", OUTPUT.resolve())


,source_sheet,raw_records,records_after_exact_dedup,included_onset_records,flag_missing_onset,excluded_records,flag_missing_or_invalid_zip,flag_zip_not_matched_to_nj_reference,repeated_person_zip_onset_records,exact_duplicate_records_removed,raw_missing_onset,flag_zip_matches_ny_reference_state_conflict
0,2020,526940,526923,265371,259237,261552,6845,1006,66,17,259251,0.0
1,2021,992532,992263,437272,553280,554991,14692,1273,177,269,553538,72.0
2,2022,957771,957047,301336,654930,655711,15398,1732,162,724,655572,30.0
3,2023,179645,179568,36415,143061,143153,525,406,21,77,143138,2.0


,state,zip,date,onset_day_of_week,onset_day_name,onset_record_count,unique_person_zip_onset_count
0,NJ,07001,2020-03-08,7,Sunday,1,1
1,NJ,07001,2020-03-09,1,Monday,1,1
2,NJ,07001,2020-03-12,4,Thursday,2,2
3,NJ,07001,2020-03-15,7,Sunday,2,2
4,NJ,07001,2020-03-16,1,Monday,1,1
5,NJ,07001,2020-03-17,2,Tuesday,3,3
6,NJ,07001,2020-03-18,3,Wednesday,2,2
7,NJ,07001,2020-03-20,5,Friday,2,2
8,NJ,07001,2020-03-21,6,Saturday,5,5
9,NJ,07001,2020-03-22,7,Sunday,4,4


Raw records: 2,656,888
Included onset records: 1,040,394
Missing onset share: 60.65%
Balanced rows: 873,678
Saved to: C:\Users\41240\OneDrive\桌面\MBS\2026 fall\Advanced Analytics\data\real_nj_cleaning_outputs


## 5. Decisions before modeling

1. Confirm the case/episode unit, reinfection treatment and meaning of `numrecs`. The current count does not sum `numrecs`.
2. Confirm geographic and time coverage before accepting zero-filled days for model training. Missing onset is not a zero-case observation.
3. Review onset missingness by ZIP and report month. Excluding unknown onset may change the trends being measured.
4. Validate postal ZIP-to-ZCTA mapping, then join population and commuting features.
5. For NJ-only data, retain only NJ destinations for forecasts and explicitly define NJ-origin pressure. Do not replace missing NY origin rates with zero.
6. Create historical features, 1–14-day targets and chronological splits after the target is agreed. Evaluate the contribution of commuting features against otherwise comparable models.
7. The original NJ-to-NY forecasting question still needs NY outcome data or an explicitly revised NJ-only scope. Stage Two will test whether COVID-derived regional patterns also appear in other respiratory diseases, at the geographic resolution actually available.
